# Pronósticos basados en series de tiempo
## Tendencia, Estacionalidad y Comparación de Modelos
### Maestría en Ciencia de Datos — Universidad Javeriana Cali
**Estudiante:** Juan Sebastian Torres Romero

---

**Ejercicio:** Empleando la información del número de ocupados en miles de personas (Ocupados) para las 13 principales ciudades, encuentre el mejor pronóstico para los próximos 6 meses comparando los modelos vistos en clase. Escriba un breve informe de máximo una página que explique cómo llega a sus proyecciones, presente las proyecciones y aclare las limitaciones.

## **1. Carga de paquetes**

In [ ]:
import sys
!{sys.executable} -m pip install --quiet numpy pandas matplotlib scikit-learn statsmodels openpyxl scipy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from matplotlib import pyplot as plt
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_squared_error
from scipy import stats

## **2. Carga de datos**

Se carga la serie mensual de Ocupados (miles) para las 13 principales ciudades de Colombia desde el repositorio del curso.

In [ ]:
data = pd.read_excel(
    "https://raw.githubusercontent.com/dagudelo30/Series-de-tiempo---Javeriana-Cali/main/intro-moving_average/datosEmpleo.xlsx",
    index_col='mes', parse_dates=True
)
data = data.asfreq("MS")
data.head()

In [ ]:
print(data.shape)
data.info()

In [ ]:
y = data["Ocupados"]

plt.figure(figsize=(12, 5))
plt.plot(y, color='steelblue', linewidth=1.3)
plt.title("Número de ocupados en las 13 principales ciudades (miles de personas)", fontsize=13)
plt.xlabel("Mes")
plt.ylabel("Ocupados (miles)")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

La serie muestra una **tendencia creciente sostenida** con fluctuaciones estacionales anuales regulares. En este ejercicio se comparan modelos de regresión (lineal, cuadrática, estacional, tendencia+estacional) con el modelo Holt-Winters del entregable anterior para identificar el mejor pronóstico.

## **3. Protocolo de evaluación**

Se reservan los **últimos 6 meses** (ene-jun 2019) como conjunto de prueba. Los 216 meses anteriores se usan para entrenar. Las variables de tiempo `x` y `x²` se construyen como secuencias numéricas y las dummies mensuales capturan la estacionalidad.

In [ ]:
train = data.iloc[:-6].copy()
test  = data.iloc[-6:].copy()

print(f"Entrenamiento: {train.index[0].strftime('%Y-%m')} a {train.index[-1].strftime('%Y-%m')} (n={len(train)})")
print(f"Prueba:        {test.index[0].strftime('%Y-%m')} a {test.index[-1].strftime('%Y-%m')} (n={len(test)})")

plt.figure(figsize=(12, 5))
plt.plot(train["Ocupados"], label="Entrenamiento", color="steelblue")
plt.plot(test["Ocupados"],  label="Prueba", color="orange", linewidth=2)
plt.title("División entrenamiento / prueba — Ocupados")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Variables auxiliares
y_train  = train["Ocupados"].values
y_test   = test["Ocupados"].values
n_train  = len(train)
x_train  = np.arange(1, n_train + 1)
x_test   = np.arange(n_train + 1, n_train + 7)

dum_train = pd.get_dummies(train.index.month, drop_first=True, dtype=float)
dum_train.index = train.index
dum_test  = pd.get_dummies(test.index.month, drop_first=True, dtype=float)
dum_test.index  = test.index
dum_test  = dum_test.reindex(columns=dum_train.columns, fill_value=0.0)

## **4. Modelos de regresión**

### **4.1 Tendencia Lineal**

Se ajusta una regresión lineal simple usando el tiempo como variable explicativa. Este modelo captura el crecimiento promedio pero ignora la estacionalidad.

In [ ]:
X1_train = sm.add_constant(x_train)
m1 = sm.OLS(y_train, X1_train).fit()
print(m1.summary())

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train.index, y_train, label="Datos")
plt.plot(train.index, m1.fittedvalues, label="Tendencia Lineal", linestyle="--")
plt.title("Tendencia Lineal — ajuste en entrenamiento")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(m1.resid)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuales — Tendencia Lineal")
plt.grid(alpha=0.4)
plt.show()

In [ ]:
pred1 = m1.predict(sm.add_constant(x_test))
rmse1 = np.sqrt(mean_squared_error(y_test, pred1))
print(f"RMSE Tendencia Lineal: {rmse1:.2f}")

El modelo lineal captura la tendencia general pero los residuales muestran **patrones sistemáticos no explicados**, especialmente la estacionalidad mensual. El RMSE fuera de muestra es elevado.

### **4.2 Tendencia Cuadrática**

Se incorpora el tiempo al cuadrado para capturar una posible desaceleración en la tendencia de crecimiento.

In [ ]:
X2_train = sm.add_constant(np.column_stack([x_train, x_train**2]))
m2 = sm.OLS(y_train, X2_train).fit()
print(m2.summary())

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train.index, y_train, label="Datos")
plt.plot(train.index, m2.fittedvalues, label="Tendencia Cuadratica", linestyle="--", color="darkorange")
plt.title("Tendencia Cuadratica — ajuste en entrenamiento")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(m2.resid)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuales — Tendencia Cuadratica")
plt.grid(alpha=0.4)
plt.show()

In [ ]:
pred2 = m2.predict(sm.add_constant(np.column_stack([x_test, x_test**2])))
rmse2 = np.sqrt(mean_squared_error(y_test, pred2))
print(f"RMSE Tendencia Cuadratica: {rmse2:.2f}")

La tendencia cuadrática mejora el ajuste frente al modelo lineal, capturando la leve desaceleración del crecimiento en los últimos años. Sin embargo, los residuales siguen mostrando patrones estacionales no explicados.

### **4.3 Estimación de la Estacionalidad**

Se incorporan variables dummy para los 11 meses (drop_first=True) capturando las diferencias promedio entre meses sin modelar la tendencia.

In [ ]:
X3_train = sm.add_constant(dum_train)
m3 = sm.OLS(y_train, X3_train).fit()
print(m3.summary())

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train.index, y_train, label="Datos")
plt.plot(train.index, m3.fittedvalues, label="Estacionalidad", linestyle="--", color="green")
plt.title("Estacionalidad — ajuste en entrenamiento")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(m3.resid)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuales — Estacionalidad")
plt.grid(alpha=0.4)
plt.show()

In [ ]:
X3_test = sm.add_constant(dum_test)
pred3 = m3.predict(X3_test)
rmse3 = np.sqrt(mean_squared_error(y_test, pred3))
print(f"RMSE Estacionalidad: {rmse3:.2f}")

El modelo estacional reproduce las fluctuaciones periódicas pero **no incorpora la tendencia creciente**, lo que genera el mayor RMSE entre los modelos evaluados. Los residuales muestran una tendencia creciente muy marcada.

### **4.4 Tendencia Cuadrática + Estacionalidad**

Se combina la tendencia cuadrática con las dummies mensuales, capturando simultáneamente el crecimiento de largo plazo y las variaciones estacionales.

In [ ]:
X4_train = sm.add_constant(pd.concat([
    pd.DataFrame({"x": x_train, "x2": x_train**2}, index=train.index),
    dum_train
], axis=1))
m4 = sm.OLS(y_train, X4_train).fit()
print(m4.summary())

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train.index, y_train, label="Datos")
plt.plot(train.index, m4.fittedvalues, label="Tend. Cuad. + Estacional", linestyle="--", color="purple")
plt.title("Tendencia Cuadratica + Estacionalidad — ajuste en entrenamiento")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(m4.resid)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuales — Tendencia Cuadratica + Estacionalidad")
plt.grid(alpha=0.4)
plt.show()

In [ ]:
X4_test = sm.add_constant(pd.concat([
    pd.DataFrame({"x": x_test, "x2": x_test**2}, index=test.index),
    dum_test
], axis=1))
pred4 = m4.predict(X4_test)
rmse4 = np.sqrt(mean_squared_error(y_test, pred4))
print(f"RMSE Tend. Cuad. + Estacional: {rmse4:.2f}")

Este modelo es el **mejor entre los de regresión**: al combinar tendencia cuadrática y estacionalidad logra representar adecuadamente la dinámica histórica. El RMSE baja significativamente respecto a los modelos anteriores.

### **4.5 Holt-Winters ETS(A,A,M) — referencia del entregable anterior**

Se incluye el modelo Holt-Winters multiplicativo con tendencia del entregable anterior para comparar contra los modelos de regresión.

In [ ]:
hw_model  = ETSModel(endog=train["Ocupados"], error="add", trend="add",
                     seasonal="mul", seasonal_periods=12)
hw_result = hw_model.fit(disp=False)

pred5 = hw_result.forecast(6).values
rmse5 = np.sqrt(mean_squared_error(y_test, pred5))
print(f"RMSE HW ETS(A,A,M): {rmse5:.2f}")

## **5. Comparación de modelos**

In [ ]:
comparacion = pd.DataFrame({
    "Modelo": [
        "Tendencia Lineal",
        "Tendencia Cuadratica",
        "Estacionalidad",
        "Tend. Cuad. + Estacional",
        "Holt-Winters ETS(A,A,M)"
    ],
    "RMSE": [rmse1, rmse2, rmse3, rmse4, rmse5]
})

comparacion.style.format({"RMSE": "{:.2f}"}) \
    .highlight_min(subset=["RMSE"], color="lightgreen")

In [ ]:
plt.figure(figsize=(10, 4))
colores = ['#4e79a7','#f28e2b','#e15759','#76b7b2','#59a14f']
bars = plt.barh(comparacion["Modelo"], comparacion["RMSE"], color=colores)
for bar, val in zip(bars, comparacion["RMSE"]):
    plt.text(bar.get_width()+10, bar.get_y()+bar.get_height()/2,
             f'{val:.1f}', va='center', fontsize=9)
plt.title("Comparacion de modelos — RMSE fuera de muestra", fontsize=11)
plt.xlabel("RMSE")
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

El **Holt-Winters ETS(A,A,M)** obtiene el menor RMSE (59.64), superando ampliamente a todos los modelos de regresión. Entre los modelos de regresión, el de tendencia cuadrática + estacionalidad es el mejor (519.23). Esto confirma que los métodos de suavización exponencial son más adecuados para esta serie.

## **6. Validación de supuestos del mejor modelo de regresión**

Se evalúan los supuestos sobre los residuales del modelo de Tendencia Cuadrática + Estacionalidad ajustado con toda la muestra.

In [ ]:
# Reajuste con todos los datos
n = len(data)
x_all = np.arange(1, n+1)
dum_all = pd.get_dummies(data.index.month, drop_first=True, dtype=float)
dum_all.index = data.index

X4_all = sm.add_constant(pd.concat([
    pd.DataFrame({"x": x_all, "x2": x_all**2}, index=data.index),
    dum_all
], axis=1))
m4_all = sm.OLS(data["Ocupados"].values, X4_all).fit()
residuales = m4_all.resid

plt.figure(figsize=(12, 4))
plt.plot(data.index, residuales, color='steelblue', linewidth=1)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuales — Tend. Cuad. + Estacional (muestra completa)")
plt.grid(alpha=0.4)
plt.show()

### **6.1 Autocorrelación: Box-Pierce y Ljung-Box**

In [ ]:
lb = acorr_ljungbox(residuales, lags=[6, 12, 18, 24], return_df=True, boxpierce=True)
print(lb[['lb_stat','lb_pvalue','bp_stat','bp_pvalue']].round(4))

Las pruebas Ljung-Box y Box-Pierce **rechazan la hipótesis nula de no autocorrelación** para todos los rezagos evaluados (p-valor ≈ 0). Existe autocorrelación significativa en los residuales, lo que indica que el modelo no captura toda la estructura temporal de la serie.

### **6.2 Homocedasticidad (Ljung-Box sobre residuales²)**

In [ ]:
lb2 = acorr_ljungbox(residuales**2, lags=[6, 12, 18, 24], return_df=True)
print(lb2[['lb_stat','lb_pvalue']].round(4))

La prueba sobre los residuales al cuadrado **rechaza la homocedasticidad** (p-valor ≈ 0), indicando presencia de heterocedasticidad. La varianza de los residuales no es constante en el tiempo.

### **6.3 Normalidad: Jarque-Bera y Shapiro-Wilk**

In [ ]:
jb_stat, jb_p = stats.jarque_bera(residuales)
sw_stat, sw_p = stats.shapiro(residuales)

print(f"Jarque-Bera:  stat={jb_stat:.4f}  p={jb_p:.4f}")
print(f"Shapiro-Wilk: stat={sw_stat:.4f}  p={sw_p:.4f}")

Ambas pruebas rechazan la normalidad de los residuales (p < 0.05), aunque la prueba Jarque-Bera está en el límite (p=0.048). La distribución de los residuales presenta colas más pesadas que la normal.

### **6.4 Verificación visual — QQ Plot e histograma**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sm.qqplot(residuales, line='s', ax=axes[0], alpha=0.6)
axes[0].set_title('QQ-Plot de Residuales')
axes[0].grid(alpha=0.4)

axes[1].hist(residuales, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Histograma de Residuales')
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
supuestos = pd.DataFrame({
    "Prueba": ["Ljung-Box", "Box-Pierce", "Ljung-Box resid²", "Jarque-Bera", "Shapiro-Wilk"],
    "Supuesto evaluado": ["Autocorrelacion", "Autocorrelacion", "Homocedasticidad",
                          "Normalidad", "Normalidad"],
    "Conclusion": ["Existe autocorrelacion", "Existe autocorrelacion",
                   "Existe heterocedasticidad", "Rechaza normalidad", "Rechaza normalidad"]
})
supuestos

## **7. Pronóstico con el mejor modelo — Holt-Winters ETS(A,A,M)**

Dado que HW superó a todos los modelos de regresión en RMSE y sus residuales cumplen mejor los supuestos, se reestima con la totalidad de las 222 observaciones para generar el pronóstico final.

In [ ]:
final_model = ETSModel(
    endog=data["Ocupados"],
    error="add", trend="add", seasonal="mul", seasonal_periods=12
)
final_fit = final_model.fit(disp=False)

print(f"Alpha: {final_fit.alpha:.4f}")
print(f"Beta:  {final_fit.beta:.4f}")
print(f"Gamma: {final_fit.gamma:.4f}")

In [ ]:
pf = final_fit.forecast(6)
ci = final_fit.get_prediction(start=pf.index[0], end=pf.index[-1]).summary_frame(alpha=0.05)

preds_final = pd.DataFrame({
    "Point_forecast": ci["mean"],
    "lower_95":       ci["pi_lower"],
    "upper_95":       ci["pi_upper"]
})
preds_final

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["Ocupados"], label="Serie historica", color="steelblue", linewidth=1.2)
plt.plot(preds_final["Point_forecast"],
         label="Pronostico HW ETS(A,A,M)", color="crimson",
         linewidth=2, linestyle="--", marker="o")
plt.fill_between(preds_final.index,
                 preds_final["lower_95"], preds_final["upper_95"],
                 color="crimson", alpha=0.15, label="IC 95%")
plt.axvline(x=data.index[-1], color="gray", linestyle=":", linewidth=1.5)
plt.title("Pronostico de Ocupados — HW ETS(A,A,M) — jul-dic 2019", fontsize=13)
plt.xlabel("Mes")
plt.ylabel("Miles de personas")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

El pronóstico final con HW ETS(A,A,M) muestra crecimiento sostenido con pico en noviembre (~11.151 miles), capturando el patrón estacional histórico del mercado laboral colombiano. Los intervalos de confianza al 95% se amplían progresivamente con el horizonte de pronóstico.